In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.DataFrame({
    "id": range(1, 101),
    "name": [f"user_{i}" for i in range(1, 101)],
    "score": np.random.uniform(0, 100, 100).round(2),
    "active": np.random.choice([True, False], 100)
})
df

,id,name,score,active
0,1,user_1,65.23,False
1,2,user_2,73.38,False
2,3,user_3,77.08,False
3,4,user_4,22.63,True
4,5,user_5,59.30,False
...,...,...,...,...
95,96,user_96,36.51,False
96,97,user_97,47.50,False
97,98,user_98,76.10,False
98,99,user_99,53.31,True


In [3]:
%%duckdb

select * from df

,id,name,score,active
0,1,user_1,65.23,False
1,2,user_2,73.38,False
2,3,user_3,77.08,False
3,4,user_4,22.63,True
4,5,user_5,59.30,False
...,...,...,...,...
95,96,user_96,36.51,False
96,97,user_97,47.50,False
97,98,user_98,76.10,False
98,99,user_99,53.31,True


In [4]:
from datetime import datetime, timedelta

import polars as pl
import pyarrow as pa


# 1. PyArrow ile In-Memory Veri Oluşturma (Sıfırdan)
tarihler = [datetime(2026, 1, 1) + timedelta(days=i) for i in range(6)]
kategoriler = pa.array(['Elektronik', 'Giyim', 'Elektronik', 'Giyim', 'Gıda', 'Gıda'])
satislar = pa.array([1500, 400, 2100, 600, 150, 300])

# PyArrow Table Oluşturma
pa_tablo = pa.Table.from_arrays(
    [pa.array(tarihler), kategoriler, satislar],
    names=['Tarih', 'Kategori', 'Satis_Miktari']
)

df_pl = pl.from_arrow(pa_tablo)

In [5]:
%%duckdb

select * from df_pl

,Tarih,Kategori,Satis_Miktari
0,2026-01-01,Elektronik,1500
1,2026-01-02,Giyim,400
2,2026-01-03,Elektronik,2100
3,2026-01-04,Giyim,600
4,2026-01-05,Gıda,150
5,2026-01-06,Gıda,300


In [6]:
%%duckdb

select * from pa_tablo

,Tarih,Kategori,Satis_Miktari
0,2026-01-01,Elektronik,1500
1,2026-01-02,Giyim,400
2,2026-01-03,Elektronik,2100
3,2026-01-04,Giyim,600
4,2026-01-05,Gıda,150
5,2026-01-06,Gıda,300


In [7]:
%%duckdb -t pl -o df

select * from generate_series(1,5)

generate_series
i64
1
2
3
4
5


In [9]:
df.count()

generate_series
u32
5


In [14]:
%%duckdb  -t pl -o pointwise_stats

select  d1.Zone as pu_zone, d2.Zone as do_zone, avg(total_amount) mean ,max(total_amount) max, median(total_amount) min, count(1) as n
from 's3://tt-bootcamp-shared/nyc-taxi-trip/green-taxi-data/*.parquet' f
JOIN 's3://tt-bootcamp-shared/nyc-taxi-trip/zone-lookup/taxi_zone_lookup.parquet' d1 on(f.PULocationID = d1.LocationID)
JOIN 's3://tt-bootcamp-shared/nyc-taxi-trip/zone-lookup/taxi_zone_lookup.parquet' d2 on(f.DOLocationID = d2.LocationID)
group by 1,2
having count(1) > 10
order by 3 desc

pu_zone,do_zone,mean,max,min,n
str,str,f64,f64,f64,i64
"""Jamaica""","""Newark Airport""",196.207692,252.65,191.03,13
"""Brooklyn Heights""","""Outside of NYC""",175.108824,961.0,116.49,17
"""East Harlem North""","""Newark Airport""",163.926667,189.19,169.2,27
"""West Concourse""","""Outside of NYC""",162.258835,1095.54,148.12,103
"""Park Slope""","""Outside of NYC""",157.083462,383.0,159.78,26
…,…,…,…,…,…
"""Bloomingdale""","""Manhattan Valley""",9.727826,25.5,9.4,368
"""Glendale""","""Glendale""",9.276308,43.2,7.0,65
"""Soundview/Bruckner""","""West Farms/Bronx River""",9.209091,11.2,9.7,11


In [15]:
pointwise_stats

pu_zone,do_zone,mean,max,min,n
str,str,f64,f64,f64,i64
"""Jamaica""","""Newark Airport""",196.207692,252.65,191.03,13
"""Brooklyn Heights""","""Outside of NYC""",175.108824,961.0,116.49,17
"""East Harlem North""","""Newark Airport""",163.926667,189.19,169.2,27
"""West Concourse""","""Outside of NYC""",162.258835,1095.54,148.12,103
"""Park Slope""","""Outside of NYC""",157.083462,383.0,159.78,26
…,…,…,…,…,…
"""Bloomingdale""","""Manhattan Valley""",9.727826,25.5,9.4,368
"""Glendale""","""Glendale""",9.276308,43.2,7.0,65
"""Soundview/Bruckner""","""West Farms/Bronx River""",9.209091,11.2,9.7,11


In [19]:
%%sh

ls -la | sort -rh | head -10

total 380
drwxrwsr-x 8 root    1000   4096 Jun 23 13:05 .
drwxrwsr-x 3 jovyan  1000   4096 Jun 22 10:02 .ruff_cache
drwxrwsr-x 3 jovyan  1000   4096 Jun 22 09:26 .jupyter
drwxrwsr-x 2 jovyan  1000   4096 Jun 23 12:59 .ipynb_checkpoints
drwxrws--- 2 root    1000  16384 Jun 22 09:25 lost+found
drwxr-sr-x 8 jovyan  1000   4096 Jun 23 12:41 .git
drwxr-sr-x 2 jovyan  1000   4096 Jun 23 12:41 .virtual_documents
drwsrws--- 1 jovyan users   4096 Jun 23 12:41 ..
-rw-rw-r-- 1 jovyan  1000  20780 Jun 23 04:41 100_SampleCode.ipynb
